In [2]:
import subprocess, sys

VLLM_PIN = "0.28.0"
AUTOAWQ_PIN = "0.2.*"
TRANSFORMERS_PIN = "5.16.1"
ACCELERATE_PIN = "1.14.0"
HTTPX_PIN = "0.28.1"
OPENAI_PIN = "3.7.0"

def pip_install(specs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *specs]
    print("installing:", " ".join(specs))
    subprocess.run(cmd, check=True)

In [3]:
pip_install([
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"autoawq=={AUTOAWQ_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
])

installing: vllm==0.28.0 transformers==5.16.1 accelerate==1.14.0 autoawq==0.2.* httpx==0.28.1 openai==3.7.0


In [4]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchaudio"], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'uninstall', '-y', 'torchaudio'], returncode=0)

In [5]:
try:
    import torchaudio
    print("still installed:", torchaudio.__version__)
except ImportError:
    print("torchaudio removed successfully")

torchaudio removed successfully


In [6]:
import os, signal, subprocess, sys

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": "8000",
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

def build_cmd(args: dict) -> list:
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    proc = subprocess.Popen(
        cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True,
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server()

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 5716, logging to /content/server.log


In [7]:
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=50):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass
        if server.poll() is not None:
            print(f"server process died with exit code {server.poll()}")
            print(tail_log())
            return False
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 50 log lines:")
    print(tail_log())
    return False

healthy = wait_for_health()

server healthy after about 153s: http://localhost:8000/v1/models -> 200


In [8]:
bench_code = '''"""Benchmark harness for the serving stack (week 3 day 5 reference)."""

from __future__ import annotations

import argparse
import asyncio
import json
import os
import statistics
import time
from dataclasses import dataclass, field
from typing import Optional

import httpx


@dataclass
class RequestResult:
    ok: bool
    ttft_s: Optional[float] = None
    latency_s: Optional[float] = None
    completion_tokens: int = 0
    error: Optional[str] = None


async def _one_request(client, base_url, model, prompt, max_tokens):
    body = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "stream": True,
        "temperature": 0.0,
    }
    url = base_url.rstrip("/") + "/v1/chat/completions"
    start = time.perf_counter()
    ttft = None
    tokens = 0
    try:
        async with client.stream("POST", url, json=body) as response:
            if response.status_code != 200:
                text = (await response.aread()).decode("utf-8", "replace")[:200]
                return RequestResult(ok=False, error=f"HTTP {response.status_code}: {text}")
            async for line in response.aiter_lines():
                if not line or not line.startswith("data: "):
                    continue
                data = line[len("data: "):]
                if data == "[DONE]":
                    break
                try:
                    chunk = json.loads(data)
                except json.JSONDecodeError:
                    continue
                delta = chunk.get("choices", [{}])[0].get("delta", {})
                content = delta.get("content")
                if content:
                    if ttft is None:
                        ttft = time.perf_counter() - start
                    tokens += 1
        latency = time.perf_counter() - start
        return RequestResult(ok=True, ttft_s=ttft, latency_s=latency, completion_tokens=tokens)
    except Exception as exc:
        return RequestResult(ok=False, error=f"{type(exc).__name__}: {exc}")


@dataclass
class LevelReport:
    concurrency: int
    tokens_per_s: float
    ttft_p50_s: Optional[float]
    ttft_p95_s: Optional[float]
    latency_p95_s: Optional[float]
    errors: int
    ok: int
    wall_s: float = field(default=0.0)


def _percentile(values, pct):
    if not values:
        return None
    ordered = sorted(values)
    if len(ordered) == 1:
        return round(ordered[0], 4)
    rank = max(1, int(round(pct / 100.0 * len(ordered))))
    rank = min(rank, len(ordered))
    return round(ordered[rank - 1], 4)


async def _run_level(client, base_url, model, prompts, concurrency, requests_per_level, max_tokens):
    await _one_request(client, base_url, model, prompts[0], max_tokens)
    semaphore = asyncio.Semaphore(concurrency)

    async def _guarded(index):
        async with semaphore:
            prompt = prompts[index % len(prompts)]
            return await _one_request(client, base_url, model, prompt, max_tokens)

    level_start = time.perf_counter()
    results = await asyncio.gather(*(_guarded(i) for i in range(requests_per_level)))
    wall = time.perf_counter() - level_start

    ok = [r for r in results if r.ok]
    errors = len(results) - len(ok)
    ttfts = [r.ttft_s for r in ok if r.ttft_s is not None]
    latencies = [r.latency_s for r in ok if r.latency_s is not None]
    total_tokens = sum(r.completion_tokens for r in ok)
    tokens_per_s = round(total_tokens / wall, 2) if wall > 0 else 0.0

    return LevelReport(
        concurrency=concurrency,
        tokens_per_s=tokens_per_s,
        ttft_p50_s=_percentile(ttfts, 50),
        ttft_p95_s=_percentile(ttfts, 95),
        latency_p95_s=_percentile(latencies, 95),
        errors=errors,
        ok=len(ok),
        wall_s=round(wall, 3),
    )


def _load_prompts(path):
    with open(path, encoding="utf-8") as handle:
        prompts = [line.strip() for line in handle if line.strip()]
    if not prompts:
        raise SystemExit(f"prompt file {path!r} has no non-empty lines")
    return prompts


def _print_table(levels):
    header = f"{{'conc':>4}}  {{'tok/s':>8}}  {{'ttft_p50':>9}}  {{'ttft_p95':>9}}  {{'lat_p95':>8}}  {{'ok':>4}}  {{'err':>4}}"
    print(header)
    print("-" * len(header))
    for lv in levels:
        def fmt(value):
            return f"{value:.3f}" if value is not None else "  n/a"
        print(f"{lv.concurrency:>4}  {lv.tokens_per_s:>8.2f}  {fmt(lv.ttft_p50_s):>9}  {fmt(lv.ttft_p95_s):>9}  {fmt(lv.latency_p95_s):>8}  {lv.ok:>4}  {lv.errors:>4}")


def _write_report(out_path, run_record):
    document = {"runs": []}
    if os.path.exists(out_path):
        try:
            with open(out_path, encoding="utf-8") as handle:
                existing = json.load(handle)
            if isinstance(existing, dict) and isinstance(existing.get("runs"), list):
                document = existing
        except (json.JSONDecodeError, OSError):
            document = {"runs": []}
    document["runs"].append(run_record)
    with open(out_path, "w", encoding="utf-8") as handle:
        json.dump(document, handle, indent=2)


async def _sweep(args):
    prompts = _load_prompts(args.prompt_file)
    concurrency_levels = [int(c) for c in args.concurrency.split(",") if c.strip()]

    timeout = httpx.Timeout(args.timeout, connect=10.0)
    limits = httpx.Limits(max_connections=max(concurrency_levels) + 4)
    levels = []

    headers = {}
    if getattr(args, "api_key", ""):
        headers["Authorization"] = "Bearer " + args.api_key
    async with httpx.AsyncClient(timeout=timeout, limits=limits, headers=headers) as client:
        for concurrency in concurrency_levels:
            report = await _run_level(
                client=client,
                base_url=args.base_url,
                model=args.model,
                prompts=prompts,
                concurrency=concurrency,
                requests_per_level=args.requests_per_level,
                max_tokens=args.max_tokens,
            )
            levels.append(report)
            print(f"[level {concurrency}] tok/s={report.tokens_per_s} ttft_p95={report.ttft_p95_s} errors={report.errors}", flush=True)

    return {
        "timestamp": int(time.time()),
        "base_url": args.base_url,
        "model": args.model,
        "requests_per_level": args.requests_per_level,
        "max_tokens": args.max_tokens,
        "prompt_file": args.prompt_file,
        "levels": [vars(lv) for lv in levels],
    }, levels


def main():
    parser = argparse.ArgumentParser(description="serving-stack benchmark harness")
    parser.add_argument("--base-url", default="http://localhost:8000")
    parser.add_argument("--model", required=True)
    parser.add_argument("--concurrency", default="1,2,4,8,16")
    parser.add_argument("--requests-per-level", type=int, default=20)
    parser.add_argument("--prompt-file", default="prompts.sample.txt")
    parser.add_argument("--out", default="bench_report.json")
    parser.add_argument("--max-tokens", type=int, default=128)
    parser.add_argument("--api-key", default=os.environ.get("API_KEY", ""))
    parser.add_argument("--timeout", type=float, default=120.0)
    args = parser.parse_args()

    run_record, levels = asyncio.run(_sweep(args))
    print()
    _print_table(levels)
    _write_report(args.out, run_record)
    print(f"\\nwrote {args.out} (run appended)")


if __name__ == "__main__":
    main()
'''

with open("bench.py", "w", encoding="utf-8") as f:
    f.write(bench_code)

print("bench.py written to:", os.path.abspath("bench.py"))

bench.py written to: /content/bench.py


In [9]:
prompts = """What is a GPU?
Define tokens per second in one line.
Explain the difference between prefill and decode in two sentences.
List three reasons decode is memory-bound rather than compute-bound.
Summarise what an inference server does for an ops team, in three short bullets.
Why does a longer prompt increase time to first token but not the per-token gap?
Describe the KV cache to a new engineer and say why it grows with context length.
Walk through what continuous batching changes versus static batching, with an example of the straggler effect it removes.
Name two things weight-only quantisation trades away in exchange for smaller memory footprint.
A user asks for the weather in Riyadh and the current time in Tokyo; describe the two tool calls you would make and the arguments for each.
Write a short runbook for rolling back a bad deployment, listing the steps in order and the check after each one.
Explain, for a non-technical manager, why a busy GPU is not the same as a productive GPU, using the utilisation trap.
Compare fp16 and int4 for serving a 1.5 billion parameter model: memory, speed, and quality, in a short paragraph each.
Give a one-sentence definition of p95 latency and say why it matters more than the average for an SLO.
Draft three sentences a platform team could send another team to describe an OpenAI-compatible endpoint they can call.
Outline the symptom, hypothesis, and measurement steps you would take when throughput is lower than expected under load.
What is PagedAttention and what problem in KV cache memory does it solve? Answer in two sentences.
Explain why the knee at the SLO, not the peak throughput, is the honest capacity number for a benchmark.
Describe how you would size the GPU memory budget for a model plus its KV cache before ever loading it.
Write a calm status update for a channel of engineers explaining that latency has risen, what you suspect, and what you are doing about it, in four sentences.
"""

with open("prompts.txt", "w", encoding="utf-8") as f:
    f.write(prompts.strip() + "\n")

print("written prompts.txt, lines:", len(prompts.strip().splitlines()))

written prompts.txt, lines: 20


In [10]:
!python bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-1.5B-Instruct-AWQ" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts.txt \
  --out bench_report.json

[level 1] tok/s=136.47 ttft_p95=0.0431 errors=0
[level 2] tok/s=266.17 ttft_p95=0.0514 errors=0
[level 4] tok/s=475.6 ttft_p95=0.0553 errors=0
[level 8] tok/s=815.56 ttft_p95=0.0754 errors=0
[level 16] tok/s=1172.44 ttft_p95=0.1417 errors=0

{'conc':>4}  {'tok/s':>8}  {'ttft_p50':>9}  {'ttft_p95':>9}  {'lat_p95':>8}  {'ok':>4}  {'err':>4}
--------------------------------------------------------------------------------------------------
   1    136.47      0.030      0.043     0.941    20     0
   2    266.17      0.036      0.051     0.946    20     0
   4    475.60      0.040      0.055     0.984    20     0
   8    815.56      0.068      0.075     1.131    20     0
  16   1172.44      0.137      0.142     1.403    20     0

wrote bench_report.json (run appended)


In [12]:
import json

levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]
for L in levels:
    print(f"c={L['concurrency']:>2}  tok/s={L['tokens_per_s']:>7.1f}  "
          f"ttft_p95={L['ttft_p95_s']:.3f}  lat_p95={L['latency_p95_s']:.3f}  "
          f"errors={L['errors']}")

TARGET_P95_S = 2.0

under = [L for L in levels if L["latency_p95_s"] <= TARGET_P95_S]
knee = max(under, key=lambda L: L["concurrency"]) if under else None
print("knee:", knee)

c= 1  tok/s=  136.5  ttft_p95=0.043  lat_p95=0.941  errors=0
c= 2  tok/s=  266.2  ttft_p95=0.051  lat_p95=0.946  errors=0
c= 4  tok/s=  475.6  ttft_p95=0.055  lat_p95=0.984  errors=0
c= 8  tok/s=  815.6  ttft_p95=0.075  lat_p95=1.131  errors=0
c=16  tok/s= 1172.4  ttft_p95=0.142  lat_p95=1.403  errors=0
knee: {'concurrency': 16, 'tokens_per_s': 1172.44, 'ttft_p50_s': 0.1369, 'ttft_p95_s': 0.1417, 'latency_p95_s': 1.4033, 'errors': 0, 'ok': 20, 'wall_s': 1.818}


In [13]:
with open("knee.json", "w") as f:
    json.dump({"target_p95_s": TARGET_P95_S,
               "knee_concurrency": knee["concurrency"] if knee else None}, f)

print("wrote knee.json:", json.load(open("knee.json")))

wrote knee.json: {'target_p95_s': 2.0, 'knee_concurrency': 16}


In [19]:
# اطبع أرقامك للنشر يدويًا على progress board
print("Knee concurrency:", knee["concurrency"] if knee else None)
print("tok/s at knee:", knee["tokens_per_s"] if knee else None)
print("p95 at knee:", knee["latency_p95_s"] if knee else None)

Knee concurrency: 16
tok/s at knee: 1172.44
p95 at knee: 1.4033


In [18]:
import os, signal
if server.poll() is None:
    os.killpg(os.getpgid(server.pid), signal.SIGTERM)
    print("server terminated")
else:
    print("server already stopped")

server already stopped


In [21]:
capacity_note = """# Capacity note (team, one page)

## The numbers

- Locked model: Qwen/Qwen2.5-1.5B-Instruct-AWQ
- Target p95 end-to-end latency (your SLO today): 2.0 seconds
- Knee concurrency (highest concurrency whose p95 is still under target):
  16 (sweep-bounded — p95 at c=16 is 1.403s, still under target, and
  throughput was still rising at the top of the sweep, so the real knee
  lies past 16; the stretch run at concurrency 32 is how we'd find it)
- Tokens per second at the knee: 1172.4
- Max sustainable request rate at the target p95: ~11.0 req/s
  (20 requests completed in 1.818s wall time at concurrency 16)

## The limiting family

Memory-bound: throughput keeps rising with concurrency but the scaling ratio
shrinks at every doubling (about 1.95x, 1.79x, 1.71x, then 1.44x from c=1
through c=16) while p95 climbs steadily and monotonically (0.941s to 1.403s).
That decelerating-throughput-plus-climbing-latency signature is the tell for
a decode memory-bandwidth ceiling (KV cache reads), not a compute ceiling —
a compute-bound stack would hold latency flatter for longer before bending.

## Why the knee, not the peak

The peak tokens/s happens past the knee, at a concurrency where p95 has
already blown through the SLO — it flatters the number by serving requests
too slowly to count. The knee is the highest concurrency we can promise while
still honoring the latency target, so it's the only number that reflects
capacity we can actually deliver.
"""

with open("capacity-note.md", "w", encoding="utf-8") as f:
    f.write(capacity_note)

import os
print("written:", os.path.abspath("capacity-note.md"))
print("exists:", os.path.exists("capacity-note.md"))

written: /content/capacity-note.md
exists: True


In [23]:
# Green-check verifier for Lab W3D5 (benchmark harness).
# Paste this as the last cell of your day-5 notebook and run it. It reads
# bench_report.json (from the harness) and capacity-note.md, and checks the
# schema, that at least four concurrency levels ran, that errors are zero or
# explained, and that the capacity note is filled in.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os, re

LEVEL_KEYS = {"concurrency", "tokens_per_s", "ttft_p50_s", "ttft_p95_s",
              "latency_p95_s", "errors"}


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    # 1) bench report
    if not os.path.exists("bench_report.json"):
        fail("bench_report.json not found; run the harness in Cell 3")
    try:
        with open("bench_report.json") as fh:
            document = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"bench_report.json is not valid JSON: {exc}")

    # bench.py appends each sweep to a "runs" list rather than overwriting, so
    # the file is a document and the thing to grade is the most recent run. A
    # bare list is also accepted, for a report assembled by hand.
    if isinstance(document, dict) and isinstance(document.get("runs"), list):
        if not document["runs"]:
            fail("bench_report.json has no runs; the harness wrote nothing")
        levels = document["runs"][-1].get("levels")
        if not isinstance(levels, list):
            fail("the most recent run in bench_report.json has no levels list")
    elif isinstance(document, list):
        levels = document
    else:
        fail("bench_report.json must be the harness output ({'runs': [...]}) "
             "or a bare list of per-level objects")
    if len(levels) < 4:
        fail(f"need at least 4 concurrency levels, found {len(levels)}")

    total_errors = 0
    for i, L in enumerate(levels):
        if not isinstance(L, dict):
            fail(f"level {i} is not an object")
        missing = LEVEL_KEYS - set(L)
        if missing:
            fail(f"level {i} missing keys: {sorted(missing)}")
        if not isinstance(L["errors"], int) or L["errors"] < 0:
            fail(f"level {i} errors must be a non-negative integer")
        total_errors += L["errors"]

    # 2) the knee file from Cell 5
    if not os.path.exists("knee.json"):
        fail("knee.json not found; write it in Cell 5")
    try:
        with open("knee.json") as fh:
            knee = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"knee.json is not valid JSON: {exc}")
    target = knee.get("target_p95_s")
    if not isinstance(target, (int, float)) or target <= 0:
        fail("target_p95_s is not a positive number; set TARGET_P95_S to your "
             "real SLO before computing the knee (the 'target left at zero' "
             "failure mode)")
    kc = knee.get("knee_concurrency")
    if not isinstance(kc, int) or kc < 1:
        fail("knee_concurrency is empty: no level stayed under your target. "
             "Either your SLO is stricter than this stack can serve (explain "
             "that in the note) or the target was never set from the card")

    # errors must be zero, OR explained in the capacity note
    # 3) capacity note filled in
    if not os.path.exists("capacity-note.md"):
        fail("capacity-note.md not found")
    with open("capacity-note.md") as fh:
        note = fh.read()
    remaining = re.findall(r"FILL:", note)
    if remaining:
        fail(f"capacity-note.md has {len(remaining)} unfilled FILL: placeholders")

    if total_errors > 0 and not re.search(r"error", note, re.I):
        fail(f"{total_errors} request errors in the sweep and no explanation in "
             "capacity-note.md; zero errors, or explain them")

    # sanity: throughput should be present and positive somewhere
    if not any(isinstance(L["tokens_per_s"], (int, float)) and L["tokens_per_s"] > 0
               for L in levels):
        fail("no level reports positive tokens_per_s")

    concurrencies = sorted(L["concurrency"] for L in levels)
    print(f"levels: {len(levels)}, concurrencies: {concurrencies}, "
          f"total errors: {total_errors}")
    print("capacity-note.md: all fields filled")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS


In [24]:
import os, signal
if server.poll() is None:
    os.killpg(os.getpgid(server.pid), signal.SIGTERM)
    print("server terminated")

In [25]:
from google.colab import files
for f_ in ["bench_report.json", "capacity-note.md", "knee.json"]:
    files.download(f_)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>